In [2]:
f_CAN = 1e6     # the bitrate you want to use for the CAN bus
f_FDCAN = 170e6 # look at the FDCAN clock frequency in the CLOCK CONFIGURATION

target_sample_point = 0.875

best = None

for prescaler in range(1, 513):
    for seg1 in range(1, 257):
        for seg2 in range(1, 129):

            total_tq = 1 + seg1 + seg2

            baud_rate = f_FDCAN / (prescaler * total_tq)

            baud_error = abs(baud_rate - f_CAN) / f_CAN

            sample_point = (1 + seg1) / total_tq
            sample_error = abs(sample_point - target_sample_point)

            score = baud_error * 1000 + sample_error

            if best is None or score < best["score"]:
                best = {
                    "prescaler": prescaler,
                    "seg1": seg1,
                    "seg2": seg2,
                    "sjw": min(seg2, 4),
                    "total_tq": total_tq,
                    "baud_rate": baud_rate,
                    "baud_error": baud_error,
                    "sample_point": sample_point,
                    "score": score
                }

print("FDCAN configuration")
print("-------------------")
print(f"FDCAN Clock:          {f_FDCAN / 1e6:.3f} MHz")
print(f"Target CAN bitrate:   {f_CAN / 1e6:.3f} Mbps")
print()
print(f"Nominal Prescaler:    {best['prescaler']}")
print(f"Nominal Time Seg1:    {best['seg1']}")
print(f"Nominal Time Seg2:    {best['seg2']}")
print(f"Nominal SJW:          {best['sjw']}")
print()
print(f"Total Time Quanta:    {best['total_tq']}")
print(f"Actual CAN bitrate:   {best['baud_rate']:.2f} bit/s")
print(f"Bitrate Error:        {best['baud_error'] * 100:.6f} %")
print(f"Sample Point:         {best['sample_point'] * 100:.2f} %")


FDCAN configuration
-------------------
FDCAN Clock:          170.000 MHz
Target CAN bitrate:   1.000 Mbps

Nominal Prescaler:    1
Nominal Time Seg1:    148
Nominal Time Seg2:    21
Nominal SJW:          4

Total Time Quanta:    170
Actual CAN bitrate:   1000000.00 bit/s
Bitrate Error:        0.000000 %
Sample Point:         87.65 %
